# mdprep — the official Colab notebook

Prepare an Amber MD system with [mdprep](https://github.com/miqueleg/mdprep),
equilibrate it, and **look at everything that was done on the way**: the
structure at each stage, the protonation states, the charge on every atom, the
parameters of a metal site, how the box equilibrated, and the MD itself.

**What you need**: a **bundle** from the
[mdprep input builder](https://miqueleg.github.io/mdprep/), the structure
and the `system.yaml` the builder wrote from your answers. Everything about
the structure is decided there: your own, from a crystal, a cofolding or a
docking model, with its substrate, cleaned, and with the crystal waters you
could bring across; or, if you have none, the SyrB2 crystal structure (PDB
2FCT) the builder loads in one click.

**First**: *Runtime → Change runtime type → T4 GPU* (or better). The
preparation runs on the CPU; the MD needs the GPU.

| Section | What it does | Time |
|---|---|---|
| 1 | Install mdprep in this notebook | 1 min |
| A | Install the programs mdprep runs (AmberTools, xTB, PySCF, OpenMM, ...) | 5–10 min |
| B | Upload your bundle | — |
| C | Prepare the system | 5–30 min |
| D | Equilibrate (Roe–Brooks protocol) and run a short production | 10–40 min |
| E | Investigate the preparation: reports, structures, protonation, charges, metal site, box | — |
| F | How the system equilibrated: temperature, density, volume, energy | — |
| G | The system after each equilibration step | — |
| H | Watch the MD: trajectory, RMSD, RMSF, metal-site distances | — |
| I | Download everything | — |

Run 1, the helpers and **A** as soon as the notebook is open: they need no
bundle, and A takes a while. Then, with your bundle, run the rest from top to
bottom. C and D do the computing; everything after them reads what they
wrote. The form fields on the right of a cell are the only things you
normally change; expand a cell (double-click) to read its code.

In [ ]:
#@title 1. Install mdprep in this notebook
#@markdown Which mdprep to install: a version number installs that release from PyPI; `master` is the repository as it is today. Use the same one the input builder used.
MDPREP_VERSION = "0.5.0"  #@param {type:"string"}

import os
import subprocess
from pathlib import Path

PREFIX = Path(os.environ.get("MDPREP_COLAB_PREFIX", "/content"))
WORK = PREFIX / "mdprep"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

if MDPREP_VERSION.strip() in ("", "master"):
    MDPREP_SPEC = '"mdprep @ git+https://github.com/miqueleg/mdprep@master"'
else:
    MDPREP_SPEC = f'"mdprep=={MDPREP_VERSION.strip()}"'
MDPREP_SPEC = os.environ.get("MDPREP_SPEC", MDPREP_SPEC)

if not os.environ.get("MDPREP_SKIP_KERNEL_INSTALL"):
    subprocess.run(f"pip install -q {MDPREP_SPEC} py3Dmol", shell=True, check=True)

import mdprep
print(f"mdprep {mdprep.__version__}; working in {WORK}")

In [ ]:
#@title Helper functions (run once; expand to read them)
import glob
import json
import shutil
import sys
import time
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import py3Dmol
import yaml
from IPython.display import Markdown, display

from mdprep.analysis import charges as charge_tables
from mdprep.analysis import equilibration as eq
from mdprep.analysis import trajectory as tr
from mdprep.analysis.metal_site import site_bonds
from mdprep.analysis.prmtop import read_prmtop
from mdprep.md.roe_brooks import STAGES
from mdprep.structure.reader import read_structure
from mdprep.structure.writer import render_pdb

IN_COLAB = "google.colab" in sys.modules
RUNTIME_NAME = "mdprep-runtime"
MICROMAMBA = PREFIX / "bin" / "micromamba"
METALS = ["Fe", "Zn", "Cu", "Mn", "Co", "Ni", "Mg", "Ca", "Mo", "V", "Cd", "Hg", "W"]
IONS = ["Cl", "Br", "I", "F", "Na", "K"]
# The counterions and salt tleap adds, named after their charge; a crystal or
# site ion keeps its own name (CL, CL1).
ADDED_IONS = ["Na+", "Cl-", "K+", "Br-", "Li+", "Rb+", "Cs+", "F-", "I-"]
WATERS = ["HOH", "WAT"]
pd.set_option("display.max_rows", 200)


def upload(kind, suffixes):
    """Ask for one file; outside Colab, MDPREP_TEST_UPLOAD_<KIND> names it."""
    preset = os.environ.get(f"MDPREP_TEST_UPLOAD_{kind.upper()}")
    if preset:
        target = WORK / Path(preset).name
        if Path(preset).resolve() != target.resolve():
            shutil.copy(preset, target)
        return target
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one file")
    name, content = next(iter(uploaded.items()))
    target = WORK / name
    target.write_bytes(content)
    if suffixes and target.suffix.lower() not in suffixes:
        raise ValueError(f"{name} is not one of {', '.join(suffixes)}")
    return target


def download(path):
    if IN_COLAB:
        from google.colab import files
        files.download(str(path))
    else:
        print("Saved", path)


def run(command, *, env=None, cwd=None):
    """Run a command, print its output as it comes, stop the notebook if it fails."""
    command = [str(part) for part in command]
    print("$", " ".join(command[-6:]) if len(command) > 6 else " ".join(command))
    process = subprocess.Popen(
        command, cwd=cwd or WORK, env=env or os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    if process.wait():
        raise RuntimeError(f"{command[0]} failed with exit code {process.returncode}; the output above says why.")


def runtime_env():
    from mdprep.workflows.runtime import gxtb_bin_directory, gxtb_executable_path
    env = {**os.environ, "MAMBA_ROOT_PREFIX": str(PREFIX / "mamba-root")}
    if Path(gxtb_executable_path(str(PREFIX))).exists():
        env["MDPREP_GXTB_EXECUTABLE"] = gxtb_executable_path(str(PREFIX))
        env["PATH"] = gxtb_bin_directory(str(PREFIX)) + ":" + env["PATH"]
    return env


def mdprep_in_runtime(*arguments):
    """mdprep inside the environment section A built, where tleap and friends live."""
    run([MICROMAMBA, "run", "-n", RUNTIME_NAME, "mdprep", *arguments], env=runtime_env())


def output_dir():
    manifest = yaml.safe_load(MANIFEST.read_text())
    return (MANIFEST.parent / manifest["project"]["output_dir"]).resolve()


def md_dir():
    return output_dir() / "md" / "roe_brooks_2020"


def pdb_text(path):
    """PDB text a viewer can read, from a PDB or an mmCIF."""
    path = Path(path)
    if path.suffix.lower() in {".cif", ".mmcif"}:
        return render_pdb(read_structure(path))
    return path.read_text()


def has_metal(text):
    return any(
        line[76:78].strip().capitalize() in METALS
        for line in text.splitlines()
        if line.startswith(("ATOM", "HETATM"))
    )


def show(text, *, title=None, site_radius=6.0, bulk_water=False, frames=False,
         width=900, height=520):
    """Protein as cartoon, ligands as sticks, metals as spheres, waters near a metal."""
    if title:
        display(Markdown(f"**{title}**"))
    view = py3Dmol.view(width=width, height=height)
    if frames:
        view.addModelsAsFrames(text, "pdb")
    else:
        view.addModel(text, "pdb")
    view.setStyle({"hetflag": False}, {"cartoon": {"color": "spectrum"}})
    view.setStyle({"hetflag": True, "not": {"resn": WATERS}}, {"stick": {"radius": 0.22}})
    view.setStyle({"elem": METALS}, {"sphere": {"radius": 0.7, "color": "darkorange"}})
    # A lone ion has no bond to draw as a stick: without a sphere, the chloride
    # on a halogenase's iron does not show at all.
    view.addStyle({"elem": IONS, "hetflag": True, "not": {"resn": ADDED_IONS}}, {"sphere": {"radius": 0.5}})
    metal_site = {"elem": METALS}
    if has_metal(text):
        view.setStyle(
            {"resn": WATERS, "within": {"distance": site_radius, "sel": metal_site}},
            {"sphere": {"radius": 0.35, "color": "red"}},
        )
        view.addStyle(
            {"byres": True, "hetflag": False, "within": {"distance": 3.0, "sel": metal_site}},
            {"stick": {"radius": 0.18}},
        )
    if bulk_water:
        view.addStyle({"resn": WATERS, "elem": "O"}, {"sphere": {"radius": 0.3, "color": "lightblue", "opacity": 0.6}})
    if frames:
        view.animate({"loop": "forward", "interval": 120})
    if has_metal(text) and not bulk_water:
        view.zoomTo({"within": {"distance": 12.0, "sel": metal_site}})
    else:
        view.zoomTo()
    view.show()


def table(rows, columns=None):
    frame = pd.DataFrame(rows)
    if columns:
        frame = frame[[column for column in columns if column in frame.columns]]
    # Numbered from 1, and nothing returned: a table on a cell's last line
    # would otherwise be shown twice.
    frame.index = range(1, len(frame) + 1)
    display(frame)


print("Helpers ready.")

## A. Install the programs mdprep runs

mdprep calls `tleap`, `antechamber`, MCPB.py, PropKa, xTB, PySCF and OpenMM.
This installs all of them, from conda-forge into an environment of their own,
and the g-xTB build of xtb from the Grimme group (its checksum is checked) —
whatever your bundle turns out to need, so it can run before you have one.
5–10 minutes.

In [ ]:
#@title A. Install the programs
#@markdown Rerun this if section D says CUDA would not start: it installs a CUDA runtime the GPU's driver can run.
from mdprep.workflows.runtime import (
    conda_packages_for, full_toolchain_manifest, gxtb_executable_path,
    gxtb_setup_commands, micromamba_install_commands, micromamba_setup_commands,
)

toolchain = full_toolchain_manifest()
print("Installing:", ", ".join(conda_packages_for(toolchain)), "and the g-xTB build of xtb")
environment = PREFIX / "mamba-root" / "envs" / RUNTIME_NAME
if environment.exists():
    # Already built in this session: bring it up to date, with the CUDA
    # runtime capped at what the GPU driver supports -- which also repairs an
    # environment whose CUDA platform would not start.
    lines = micromamba_install_commands(toolchain, environment_name=RUNTIME_NAME, prefix=str(PREFIX))
    if not Path(gxtb_executable_path(str(PREFIX))).exists():
        lines += gxtb_setup_commands(str(PREFIX))
else:
    lines = micromamba_setup_commands(toolchain, mdprep_spec=MDPREP_SPEC, prefix=str(PREFIX))
started = time.time()
run(["bash", "-c", "\n".join(lines)])
print(f"Programs ready in {(time.time() - started) / 60:.1f} min")

## B. Upload your bundle

The `mdprep-bundle.zip` the [input builder](https://miqueleg.github.io/mdprep/)
downloaded: your structure and the `system.yaml` it wrote from your answers.

In [ ]:
#@title B. Upload your bundle
if not MICROMAMBA.exists():
    raise RuntimeError("Run section A first: it installs the programs mdprep runs.")

bundle = upload("bundle", (".zip",))
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(WORK)
MANIFEST = WORK / "system.yaml"
if not MANIFEST.exists():
    raise FileNotFoundError(f"{bundle.name} has no system.yaml: download the bundle from the input builder again")

from mdprep.config.loader import load_manifest
config = load_manifest(MANIFEST, resolve_paths=True)
structure_path = Path(config.project.input_structure)
summary = [
    ("Structure", structure_path.name),
    ("Chains kept", ", ".join(config.structure.keep_chains) or "all"),
    ("Crystal waters / ions kept", f"{config.structure.keep_crystal_waters} / {config.structure.keep_crystal_ions}"),
    ("Force field / water", f"{config.protein.forcefield} / {config.protein.water_model}"),
    ("Protonation", f"{config.protonation.method} at pH {config.protonation.ph}"),
    ("Ligands", ", ".join(f"{l.id} ({l.net_charge:+d}, {l.charge_method})" for l in config.ligands) or "none"),
    ("Metal sites", ", ".join(f"{m.id} ({m.model})" for m in config.metals) or "none"),
    ("Box", f"{config.solvation.box}, {config.solvation.buffer_angstrom} A, {config.solvation.salt_concentration_molar} M salt"
            if config.solvation.enabled else "not solvated"),
]
table([{"setting": key, "value": value} for key, value in summary])
show(pdb_text(structure_path), title=f"{structure_path.name}, as it goes in")
# Every program this manifest calls, found where section A put it.
mdprep_in_runtime("doctor", MANIFEST.name)

## C. Prepare the system

`mdprep prepare` runs every stage the manifest asks for and records each
external command, its output and its timing under the output directory. It
stops **before** the MD, which section D runs in its own cell; section E shows
what was built.

In [ ]:
#@title C. Prepare
started = time.time()
mdprep_in_runtime("prepare", MANIFEST.name, "--overwrite", "--stop-after", "tleap")
print(f"Prepared in {(time.time() - started) / 60:.1f} min")
OUT = output_dir()
# Re-read the finished topology rather than trust the build.
mdprep_in_runtime("validate", OUT / "final" / "system.prmtop", OUT / "final" / "system.inpcrd")

## D. Equilibrate and run

The [Roe and Brooks (2020)](https://doi.org/10.1063/5.0013849) protocol takes
the system from tleap's box to an equilibrated one in ten stages: minimise and
heat with the solute restrained, release the restraints step by step, then let
the box find its density under constant pressure until the density stops
drifting (step 10). A short production run follows.

In [ ]:
#@title D. Equilibrate (Roe-Brooks) and run a short production
#@markdown Production after the equilibration, in picoseconds. 0 keeps what the manifest says. A few hundred ps is enough to see the system move; science needs far more.
PRODUCTION_PS = 200  #@param {type:"number"}
PLATFORM = "auto"  #@param ["auto", "CUDA", "OpenCL", "CPU"]
#@markdown `auto` tries CUDA, then OpenCL, then the CPU, and says why it passed one over. If CUDA does not start on a GPU runtime, rerun section A: it installs a CUDA runtime the GPU's driver can run.

table([{"stage": stage, "what it does": what} for stage, what in STAGES.items()])
data = yaml.safe_load(MANIFEST.read_text())
md = data.get("molecular_dynamics") or {}
md = {key: value for key, value in md.items() if key != "enabled"}
production = dict(md.get("production") or {})
if PRODUCTION_PS > 0 or not production:
    steps = int(round((PRODUCTION_PS or 200) * 1000 / 2.0))
    production = {"steps": steps, "timestep_fs": 2.0,
                  "trajectory_interval_steps": max(1, steps // 100),
                  "state_interval_steps": max(1, steps // 200),
                  "checkpoint_interval_steps": steps}
data["molecular_dynamics"] = {**md, "enabled": True, "platform": PLATFORM, "production": production}
MD_MANIFEST = MANIFEST.with_name("system.md.yaml")
MD_MANIFEST.write_text(yaml.safe_dump(data, sort_keys=False))
started = time.time()
mdprep_in_runtime("run-md", MD_MANIFEST.name)
print(f"MD finished in {(time.time() - started) / 60:.1f} min")

## E. Investigate the preparation

Every stage wrote a report. Look at them in order — this is where the
decisions you made in the input builder became a system.

In [ ]:
#@title E1. What each stage did
REPORTS = {
    "structure": "chains, crystal waters and ions kept or removed, heterogens, repairs",
    "protonation": "the state of every titratable residue, and why",
    "refinement": "the QM/MM optimisation of the active site, if there is one",
    "ligand": "how each ligand was parameterised and charged",
    "metal": "the metal sites: model, bonds, MCPB.py steps",
    "tleap": "force fields, ligands, bonds, 1-4 pairs through a metal, the box",
    "validation": "the finished topology re-read: charges, atoms, energy",
}
OUT = output_dir()
rows = []
for name, what in REPORTS.items():
    report_json = OUT / "reports" / f"{name}_report.json"
    if not report_json.exists():
        rows.append({"stage": name, "records": what, "warnings": "not run"})
        continue
    data = json.loads(report_json.read_text())
    warnings = data.get("warnings") or [] if isinstance(data, dict) else []
    rows.append({"stage": name, "records": what, "warnings": len(warnings)})
table(rows)

In [ ]:
#@title E2. Read a report
REPORT = "structure"  #@param ["structure", "protonation", "refinement", "ligand", "metal", "tleap", "validation"]
path = output_dir() / "reports" / f"{REPORT}_report.md"
display(Markdown(path.read_text()) if path.exists() else Markdown(f"*{path.name} was not written: that stage did not run.*"))

In [ ]:
#@title E3. The structure at each stage of the preparation
#@markdown `00` is the input after cleaning, `01` with hydrogens and protonation states, `02` after the QM/MM refinement of the active site (bonded metal sites only), and `final` is what tleap built — solvated, with ions.
STAGE = "01_protonation_assigned"  #@param ["00_input_normalized", "01_protonation_assigned", "02_qmmm_refined_hydrogenated", "03_final_protonation_assigned", "final"]
OUT = output_dir()
if STAGE == "final":
    # Drawn from the topology, as E8 and G are: tleap's PDB has no element
    # column, and a viewer then guesses elements from atom names -- the
    # hydrogen HG21 as mercury, bonded to everything near it.
    built = tr.load_structure(OUT / "final" / "system.prmtop", OUT / "final" / "system.inpcrd")
    atoms = tr.display_atoms(built.topology, built.positions[0])
    show(tr.pdb_models(built, atoms, [0]),
         title="final: what tleap built (the water away from the metal is hidden; E8 shows the box)")
else:
    path = OUT / "intermediate" / f"{STAGE}.pdb"
    if not path.exists():
        print(f"{path.name} does not exist for this system: that stage did not run.")
    else:
        show(path.read_text(), title=f"{path.name}")

In [ ]:
#@title E4. Protonation states
#@markdown Every residue whose state was decided, with the reason. A histidine bound to a metal through NE2 has to be HID (H on ND1); a catalytic aspartate you set by hand is listed as such.
csv_path = output_dir() / "reports" / "protonation_report.csv"
states = pd.read_csv(csv_path)
display(states[states["final_resname"].isin(["HID", "HIE", "HIP", "ASH", "GLH", "LYN", "CYM", "CYX"]) | states["changed"]])

In [ ]:
#@title E5. The charge on every atom
#@markdown A residue usually carries the integer charge of its force-field template. A **bonded metal site** does not: MCPB.py fits one set of RESP charges to the metal and everything bound to it, so only the site as a whole has the integer charge of its parts.
OUT = output_dir()
topology = read_prmtop(OUT / "final" / "system.prmtop")
totals = charge_tables.summary(topology)
print(f"Total charge of the system: {totals['total']:+.4f} e (it must be an integer, and with neutralize: true, zero)")
table([{"kind": kind, "residues": totals["residues_by_kind"][kind], "net charge": charge}
       for kind, charge in totals["by_kind"].items()])
print("Ions:", ", ".join(f"{count} {name}" for name, count in totals["ions"].items()) or "none")

# One table per site: a dimer has one in each copy, each an integer charge.
for site in charge_tables.metal_sites(topology):
    if not site["bonds"]:
        # A nonbonded ion -- a peroxygenase's magnesium -- has its formal
        # charge and nothing bonded to it.
        display(Markdown(f"**{', '.join(site['metals'])}** — a nonbonded ion, {site['total']:+.3f} e"))
        continue
    display(Markdown(f"**The metal site of {', '.join(site['metals'])}** — net charge {site['total']:+.3f} e"))
    table(site["residues"])
    table(site["bonds"], ["metal", "metal_charge", "donor", "donor_type", "donor_charge"])

ligand_residues = [r.index for r in charge_tables.residue_charges(topology) if r.kind in ("ligand", "metal_site")]
if ligand_residues:
    display(Markdown("**Every atom of the ligands and the metal site**"))
    table(charge_tables.atom_charges(topology, ligand_residues))

In [ ]:
#@title E6. The charges in 3D
#@markdown Ligands and the metal site, coloured by atomic charge: red negative, white neutral, blue positive. The metal, the atoms bound to it, and every atom carrying more than half a charge are labelled.
OUT = output_dir()
built = tr.load_structure(OUT / "final" / "system.prmtop", OUT / "final" / "system.inpcrd")
topology = built.topology
residues = [r.index for r in charge_tables.residue_charges(topology) if r.kind in ("ligand", "metal_site")]
if not residues:
    print("No ligand or metal site in this system.")
# One view per active site: the two sites of a dimer, framed together, are too far apart to read.
for group in tr.residue_clusters(topology, built.positions[0], residues):
    atoms = [index for residue in group for index in topology.residues[residue].atom_indices]
    nearby = tr.residues_near(topology, built.positions[0], atoms, 6.0)
    context = [i for r in nearby if r not in group and topology.residues[r].name not in WATERS
               for i in topology.residues[r].atom_indices]
    selection = atoms + context
    charges_b = [topology.atoms[i].charge if i in atoms else 0.0 for i in selection]
    text = tr.pdb_models(built, selection, [0], bfactors=charges_b)
    view = py3Dmol.view(width=900, height=560)
    view.addModel(text, "pdb")
    view.setStyle({}, {"line": {"color": "grey", "opacity": 0.5}})
    view.setStyle({"resi": [topology.atoms[i].residue_index + 1 for i in atoms]},
                  {"stick": {"colorscheme": {"prop": "b", "gradient": "rwb", "min": -1.0, "max": 1.0}},
                   "sphere": {"scale": 0.25, "colorscheme": {"prop": "b", "gradient": "rwb", "min": -1.0, "max": 1.0}}})
    bound = {atom for pair in topology.metal_bonds() for atom in pair}
    for index in atoms:
        atom = topology.atoms[index]
        if index in bound or abs(atom.charge) >= 0.5:
            x, y, z = built.positions[0][index]
            view.addLabel(f"{atom.charge:+.2f}", {"position": {"x": float(x), "y": float(y), "z": float(z)},
                                                    "fontSize": 10, "backgroundOpacity": 0.6})
    view.zoomTo({"resi": [topology.atoms[i].residue_index + 1 for i in atoms]})
    display(Markdown("**" + ", ".join(f"{topology.residues[r].name} {r + 1}" for r in group) + "**"))
    view.show()

In [ ]:
#@title E7. The metal site's parameters
#@markdown MCPB.py fits a force constant to each bond to the metal from a QM Hessian (Seminario method). A soft bond is one the site can stretch; compare each with its distance in the built system.
OUT = output_dir()
frcmods = sorted(glob.glob(str(OUT / "metals" / "*" / "mcpb" / "*_mcpbpy.frcmod")))
if not frcmods:
    print("No bonded (MCPB) metal site in this system.")
else:
    topology = read_prmtop(OUT / "final" / "system.prmtop")
    built = tr.load_structure(OUT / "final" / "system.prmtop", OUT / "final" / "system.inpcrd")
    rows = site_bonds(frcmods[0], topology)
    pairs = topology.metal_bonds()
    distances = dict(zip(pairs, tr.distances(built, pairs)[0]))
    labels = {(topology.label(m), topology.label(d)): distances[(m, d)] for m, d in pairs}
    for row in rows:
        row["built_A"] = round(float(labels.get((row["metal"], row["donor"]), float("nan"))), 3)
    table(rows, ["metal", "donor", "types", "k_kcal_mol_A2", "r0_A", "built_A"])
    report = (OUT / "reports" / "tleap_report.md").read_text()
    section = report.split("## 1-4 Pairs Through The Metal Switched Off")[1].split("\n## ")[0]
    display(Markdown("**1-4 pairs through the metal switched off**" + section))

In [ ]:
#@title E8. The solvated box
#@markdown The protein in its box of water (thin blue lines) and ions. A truncated octahedron is used because it needs about 30% less water than a cube for the same distance between periodic copies.
OUT = output_dir()
built = tr.image(tr.load_structure(OUT / "final" / "system.prmtop", OUT / "final" / "system.inpcrd"))
topology = built.topology
totals = charge_tables.summary(topology)
counts = totals["residues_by_kind"]
print(f"{len(topology.atoms)} atoms: {counts.get('water', 0)} waters, "
      + ", ".join(f"{count} {name}" for name, count in totals["ions"].items()))
# The oxygen of every water is enough to see the box, at a tenth of the size.
atoms = tr.display_atoms(topology, built.positions[0], bulk_water=True)
show(tr.pdb_models(built, atoms, [0]), title="The solvated system: water oxygens in light blue", bulk_water=True)

## F. How it equilibrated

Each panel is the whole equilibration on one time axis; the shaded bands are
the stages. Watch for: the **temperature** reaching 300 K in step 2 and
staying there; the **density** rising once the barostat is on (step 7) from
tleap's under-packed box to about 1.0 g/mL; the **volume** shrinking with it;
the **potential energy** settling. Step 10 runs until the density stops
drifting — its criteria are printed below.

In [ ]:
#@title F. Temperature, density, volume and energy through the equilibration
rows = eq.equilibration_table(md_dir())
frame = pd.DataFrame(rows)
spans = eq.stage_spans(rows)
panels = [("temperature_k", "Temperature (K)"), ("density_g_ml", "Density (g/mL)"),
          ("box_volume_nm3", "Box volume (nm$^3$)"), ("potential_energy_kj_mol", "Potential energy (kJ/mol)")]
panels = [panel for panel in panels if panel[0] in frame.columns]
# Minimisations have no temperature to plot; dynamics are sampled every 0.5 ps.
dynamics = frame[frame["stage_time_ps"] > 0] if "stage_time_ps" in frame else frame
early = [span for span in spans if span["stage"] not in ("step10", "production")]
early_end = max((span["end_ps"] for span in early), default=0.0) or float(frame["time_ps"].max())
windows = [(0.0, early_end, "Steps 1-9: heating, then releasing the restraints"),
           (0.0, float(frame["time_ps"].max()), "All of it: step 10 runs until the density stops drifting")]
fig, axes = plt.subplots(len(panels), 2, figsize=(13, 2.3 * len(panels)), sharex="col",
                         gridspec_kw={"width_ratios": [1, 1.6]}, squeeze=False)
for row, (column, label) in enumerate(panels):
    for col, (low, high, heading) in enumerate(windows):
        axis = axes[row, col]
        part = dynamics[(dynamics["time_ps"] >= low) & (dynamics["time_ps"] <= high)]
        axis.plot(part["time_ps"], part[column], lw=1.1)
        for index, span in enumerate(spans):
            if span["end_ps"] > span["start_ps"] and index % 2:
                axis.axvspan(span["start_ps"], min(span["end_ps"], high), color="grey", alpha=0.1)
        if col == 0:
            axis.set_ylabel(label)
        if row == 0:
            axis.set_title(heading, fontsize=10)
for span in early:
    if span["end_ps"] - span["start_ps"] >= 1.0:
        axes[0, 0].annotate(span["stage"].replace("step", ""), ((span["start_ps"] + span["end_ps"]) / 2, 0.04),
                            xycoords=("data", "axes fraction"), ha="center", fontsize=8, color="0.3")
for axis in axes[-1]:
    axis.set_xlabel("Simulated time (ps)")
plt.tight_layout()
plt.show()

report = eq.md_report(md_dir())
convergence = report["step10_convergence"]
print(f"Step 10 ran {report['step10_elapsed_ps']:.0f} ps on {report['platform_selected']}. Density over the last "
      f"{convergence['window_ps']:.0f} ps: {convergence['mean_second_half_g_ml']:.4f} g/mL, drift "
      f"{convergence['slope_g_ml_ps']:.2e} g/mL/ps (limit {convergence['slope_threshold_g_ml_ps']:.0e}), halves differ by "
      f"{convergence['mean_difference_g_ml']:.4f} (limit {convergence['mean_difference_threshold_g_ml']}).")
table([{"stage": stage["stage"], "potential (kJ/mol)": round(stage["potential_energy_kj_mol"]),
        "volume (nm3)": round(stage["box_volume_nm3"], 1), "what": STAGES.get(stage["stage"], "")}
       for stage in report["stages"]])

In [ ]:
#@title F2. The production run
log = pd.DataFrame(eq.production_table(md_dir()))
columns = [c for c in ("Temperature (K)", "Density (g/mL)", "Potential Energy (kJ/mole)") if c in log.columns]
fig, axes = plt.subplots(1, len(columns), figsize=(4 * len(columns), 3))
for axis, column in zip(np.atleast_1d(axes), columns):
    axis.plot(log["Time (ps)"], log[column])
    axis.set_title(column, fontsize=10)
    axis.set_xlabel("Time (ps)")
plt.tight_layout()
plt.show()

## G. The system after each step

Every stage wrote the structure it ended with, imaged into the box the
manifest asked for. Step through them and watch the active site: the
restraints hold the solute while the water relaxes around it, and from step 9
everything moves.

In [ ]:
#@title G. The structure after a step, and the metal site through all of them
STEP = "step10"  #@param ["step01", "step01b", "step02", "step03", "step04", "step05", "step05b", "step05c", "step06", "step07", "step08", "step09", "step10", "production_last"]
SHOW_ALL_WATER = False  #@param {type:"boolean"}

from mdprep.structure.pdb import read_pdb

topology = read_prmtop(output_dir() / "final" / "system.prmtop")
path = md_dir() / f"{STEP}.pdb"
display(Markdown(f"**{STEP}** — {STAGES.get(STEP, 'end of the production run')}"))
# The snapshot has the topology's atoms in the topology's order.
xyz = np.array([(a.x, a.y, a.z) for a in read_pdb(path).atoms])
if len(xyz) != len(topology.atoms):
    raise ValueError(f"{path.name} has {len(xyz)} atoms and the topology {len(topology.atoms)}")
snapshot = tr.Trajectory(topology=topology, positions=xyz[None], cells=None, times_ps=np.zeros(1))
atoms = tr.display_atoms(topology, xyz, bulk_water=SHOW_ALL_WATER)
show(tr.pdb_models(snapshot, atoms, [0]), bulk_water=SHOW_ALL_WATER)

pairs = topology.metal_bonds()
if pairs:
    rows = []
    for name in ["step01", "step02", "step05", "step06", "step07", "step08", "step09", "step10", "production_last"]:
        snapshot = md_dir() / f"{name}.pdb"
        if not snapshot.exists():
            continue
        atoms = read_pdb(snapshot).atoms
        xyz = np.array([(a.x, a.y, a.z) for a in atoms])
        row = {"step": name}
        for metal, donor in pairs:
            row[topology.label(donor)] = round(float(np.linalg.norm(xyz[metal] - xyz[donor])), 2)
        rows.append(row)
    display(Markdown("**Distance from the metal to each donor (A)**"))
    table(rows)

## H. Watch the MD

The production trajectory, imaged about the protein and fitted on its
C-alpha atoms. **RMSD** is how far the structure has moved from the first
frame; **RMSF** how much each residue fluctuates about its average position —
loops and termini move most. For a bonded metal site, the distance to each
donor should fluctuate about its equilibrium value without drifting away.

In [ ]:
#@title H. Trajectory, RMSD, RMSF and the metal site
#@markdown Residues and waters within this distance of a metal (in the first frame) are drawn whole; elsewhere only the backbone.
SITE_RADIUS = 6.0  #@param {type:"number"}

out = output_dir()
traj = tr.image(tr.load_trajectory(out / "final" / "system.prmtop", md_dir() / "production.dcd"))
topology = traj.topology
calpha = tr.protein_atoms(topology, {"CA"})
traj = tr.superpose(traj, calpha)
print(f"{traj.frames} frames, {traj.times_ps[0]:.0f}-{traj.times_ps[-1]:.0f} ps")

metals = [atom.index for atom in topology.atoms if topology.is_metal(atom.index)]
shown = tr.display_atoms(topology, traj.positions[0], near_metal=SITE_RADIUS, protein="backbone")
# At most 30 frames: each one is a copy of every atom drawn, in the page.
step = max(1, -(-traj.frames // 30))
show(tr.pdb_models(traj, shown, range(0, traj.frames, step)), frames=True,
     title="Production (animated): backbone, ligands, and the metal site with its waters")

panels = 3 if metals else 2
fig, axes = plt.subplots(1, panels, figsize=(5 * panels, 3.4))
axes[0].plot(traj.times_ps, tr.rmsd(traj, calpha))
axes[0].set(xlabel="Time (ps)", ylabel="C-alpha RMSD (A)", title="RMSD to the first frame")
fluctuation = tr.rmsf(traj, calpha)
axes[1].plot(np.arange(1, len(calpha) + 1), fluctuation)
axes[1].set(xlabel="Residue (C-alpha index)", ylabel="RMSF (A)", title="Per-residue fluctuation")
if metals:
    pairs = topology.metal_bonds()
    if pairs:
        series = tr.distances(traj, pairs)
        for column, (metal, donor) in enumerate(pairs):
            axes[2].plot(traj.times_ps, series[:, column], label=topology.label(donor))
        axes[2].legend(fontsize=7)
    axes[2].set(xlabel="Time (ps)", ylabel="Distance to the metal (A)", title="Metal site")
plt.tight_layout()
plt.show()

## I. Download everything

The output directory holds the final `system.prmtop`/`system.inpcrd`, every
intermediate structure, every report, every external command with its log, and
the MD. Colab's files are deleted when the session ends.

In [ ]:
#@title I. Download the results
#@markdown Restart files (`.chk`, `.xml`) let OpenMM continue this MD elsewhere. They are a quarter of the archive and not needed to look at the run again.
INCLUDE_RESTART_FILES = False  #@param {type:"boolean"}


def is_restart(path):
    return path.suffix == ".chk" or (path.suffix == ".xml" and path.name.startswith("after_"))


archive = PREFIX / "mdprep-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as results:
    for path in sorted(WORK.rglob("*")):
        # Uploaded bundles and earlier archives would only nest inside it.
        if path.is_dir() or path.suffix == ".zip":
            continue
        if is_restart(path) and not INCLUDE_RESTART_FILES:
            continue
        results.write(path, path.relative_to(WORK))
print(archive, f"{archive.stat().st_size / 1e6:.0f} MB")
download(archive)